# Fase 2 · Arquitecturas

**Detección de exoplanetas en Kepler con una red multirrama — iteración 2** (más datos, entrenamiento en GPU)

Curso de Deep Learning · Especialización en Inteligencia Artificial · Universidad Nacional de Colombia

**Autores:** Thomas Monnier Granda, Esteban Felipe Genes Noya, Juan Sebastian Higuita Riascos, Alexandra Arias Mazo y Eliel Segura Perez

> **Pregunta de este cuaderno:** ¿qué red lee cada entrada, por qué tiene esa forma y cómo se juntan en una sola respuesta?

Este cuaderno no entrena nada: lee las evidencias que dejaron los pasos 1 a 5 (`iteracion_2/*.bat`) y las explica. Cada cifra sale de un archivo del repositorio.

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import apoyo  # iteracion_2/notebooks/apoyo.py: rutas, lectura de evidencias y figuras

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
apoyo.estilo()
print("Raíz del repositorio:", apoyo.RAIZ.name, "| evidencias en", apoyo.EVIDENCIAS.relative_to(apoyo.RAIZ))

Raíz del repositorio: Proyecto_Final | evidencias en iteracion_2/evidencias


## 1. Por qué varias ramas

Una señal de tránsito se puede mirar de varias maneras, y cada una delata un tipo distinto de impostor:

| Rama | Qué mira | Qué puede delatar | Red |
|---|---|---|---|
| 1 · vistas | La forma media del tránsito y la órbita completa | Eclipses secundarios, tránsitos en V | CNN 1D de dos torres (tipo AstroNet, Shallue y Vanderburg 2018) |
| 2 · tránsitos | Cada tránsito por separado | Tránsitos que cambian de uno a otro, artefactos aislados | Codificador compartido por tránsito + agregador |
| 3 · imagen | De qué píxel viene la señal | Estrellas vecinas contaminantes | CNN 2D (curso de Visión; **no** se usa aquí) |
| 4 · escalares | Periodo, duración, profundidad, centroide | Combinaciones físicamente raras | Red densa pequeña |

Esta iteración entrena las ramas 1, 2 y 4. La rama 3 pertenece al curso de Visión por Computador y
queda **declarada como ausente** (bandera en cero), de modo que pueda sumarse después sin cambiar nada.

## 2. La rama 2 y sus tres agregadores

La rama 2 aplica la **misma** red pequeña a cada tránsito y luego tiene que resumir hasta 32 vectores
en uno. Se comparan tres formas de hacerlo:

- **Promedio:** todos los tránsitos pesan igual. No depende del orden.
- **GRU bidireccional:** lee los tránsitos en orden temporal; puede notar cambios a lo largo del tiempo.
- **Atención:** aprende cuánto pesa cada tránsito; puede ignorar los malos.

A los tres se les añade un vector de **consistencia** (por ejemplo, diferencia entre tránsitos pares e
impares: en un sistema binario eclipsante los pares y los impares suelen tener distinta profundidad).

In [2]:
filas = []
for red, nombre in apoyo.REDES.items():
    cfg = apoyo.configuraciones(red)
    filas.append({"red": nombre, "parámetros (protocolo)": int(cfg.loc[0, "parametros"]),
                  "parámetros (mayor probada)": int(cfg["parametros"].max()),
                  "vector de salida": 16 if red == "m4_escalares" else 32})
tamanos = pd.DataFrame(filas).set_index("red")
display(tamanos)

,parámetros (protocolo),parámetros (mayor probada),vector de salida
red,,,
Rama 4 · escalares,1059,142659,16
Rama 1 · vistas plegadas,57747,3451075,32
Rama 2 · promedio,5027,215971,32
Rama 2 · atención,13604,65572,32
Rama 2 · GRU,18723,347555,32


Las redes del protocolo son pequeñas a propósito (se diseñaron para 245 señales). Con 4.547 señales
la búsqueda del cuaderno 3 prueba redes decenas de veces mayores para ver si los datos nuevos las aprovechan.

## 3. Resumen de capas (opcional)

Si Keras está instalado, la celda construye las redes y muestra sus capas. Si no, se salta: el resto
del cuaderno no lo necesita.

In [3]:
try:
    import os, sys
    os.environ.setdefault("KERAS_BACKEND", "torch")
    sys.path.insert(0, str(apoyo.RAIZ / "src"))
    from temporal import modelos
    for red in (modelos.m4_escalares(0), modelos.m1_astronet(0), modelos.m2(0, "atencion", plegado=True)):
        red.summary(line_length=90)
except Exception as error:  # sin Keras o sin backend: no es un fallo del cuaderno
    print("Resumen de capas omitido:", type(error).__name__, "-", str(error)[:100])

Model: "m4_escalares"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape         ┃      Param # ┃ Connected to          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ escalares (InputLayer)   │ (None, 7)            │            0 │ -                     │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ faltantes (InputLayer)   │ (None, 7)            │            0 │ -                     │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ concatenate              │ (None, 14)           │            0 │ escalares[0][0],      │
│ (Concatenate)            │                      │              │ faltantes[0][0]       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ dense (Dense)            │ (None, 32)           │          480 │ concatenate[0][0]     │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ dropout (Dropout)        │ (None, 32)           │            0 │ dense[0][0]           │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ embedding (Dense)        │ (None, 16)           │          528 │ dropout[0][0]         │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ clases (Dense)           │ (None, 3)            │           51 │ embedding[0][0]       │
└──────────────────────────┴──────────────────────┴──────────────┴───────────────────────┘

 Total params: 1,059 (4.14 KB)

 Trainable params: 1,059 (4.14 KB)

 Non-trainable params: 0 (0.00 B)

Model: "m1_astronet_max"

... (5 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape         ┃      Param # ┃ Connected to          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ transitos (InputLayer)   │ (None, 32, 61)       │            0 │ -                     │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ plegar_transitos         │ (None, 61, 1)        │            0 │ transitos[0][0]       │
│ (PlegarTransitos)        │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ phi (Sequential)         │ (None, 32)           │        3,744 │ plegar_transitos[0][… │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ phi_por_transito         │ (None, 32, 32)       │            0 │ phi[0][0]             │
│ (DesplegarTransitos)     │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ mascara_transitos        │ (None, 32)           │            0 │ -                     │
│ (InputLayer)             │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ aplicar_mascara          │ (None, 32, 32)       │            0 │ phi_por_transito[0][… │
│ (AplicarMascara)         │                      │              │ mascara_transitos[0]… │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ mascara_booleana         │ (None, 1, 32)        │            0 │ mascara_transitos[0]… │
│ (MascaraBooleana)        │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ autoatencion             │ (None, 32, 32)       │        4,224 │ aplicar_mascara[0][0… │
│ (MultiHeadAttention)     │                      │              │ aplicar_mascara[0][0… │
│                          │                      │              │ mascara_booleana[0][… │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ add (Add)                │ (None, 32, 32)       │            0 │ aplicar_mascara[0][0… │
│                          │                      │              │ autoatencion[0][0]    │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ layer_normalization      │ (None, 32, 32)       │           64 │ add[0][0]             │
│ (LayerNormalization)     │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ dense_4 (Dense)          │ (None, 32, 64)       │        2,112 │ layer_normalization[… │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ dense_3 (Dense)          │ (None, 32, 32)       │        2,080 │ dense_4[0][0]         │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ add_1 (Add)              │ (None, 32, 32)       │            0 │ layer_normalization[… │
│                          │                      │              │ dense_3[0][0]         │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ layer_normalization_1    │ (None, 32, 32)       │           64 │ add_1[0][0]           │
│ (LayerNormalization)     │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ aplicar_mascara_1        │ (None, 32, 32)       │            0 │ layer_normalization_… │
│ (AplicarMascara)         │                      │              │ mascara_transitos[0]

 Total params: 13,604 (53.14 KB)

 Trainable params: 13,604 (53.14 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Entrenar en GPU

El requisito de esta iteración era entrenar en la GPU del equipo. Se usó Keras sobre PyTorch con CUDA.
La celda muestra el entorno que quedó registrado y una medición hecha antes de la búsqueda.

In [4]:
entorno = apoyo.leer_json(apoyo.EVIDENCIAS / "entorno.json")
display(pd.Series(entorno).to_frame("entorno de entrenamiento"))
humo = apoyo.leer_json(apoyo.EVIDENCIAS / "prueba_de_humo.json")
plegado = pd.DataFrame(humo["plegado"]).T
plegado["aceleración"] = plegado["time_distributed"] / plegado["plegado"]
display(plegado.rename(columns={"time_distributed": "s/época, tránsito a tránsito", "plegado": "s/época, todos a la vez"}).round(2))

,entorno de entrenamiento
python,3.13.0
sistema,Windows-11-10.0.26300-SP0
keras,3.15.1
backend,torch
torch,2.6.0+cu124
cuda_disponible,True
gpu,NVIDIA GeForce RTX 3060 Ti
memoria_gpu_gb,8.0
segundos_por_epoca_cpu,4.906
segundos_por_epoca_gpu,1.777


,"s/época, tránsito a tránsito","s/época, todos a la vez",aceleración
promedio,0.49,0.11,4.41
gru,1.35,0.85,1.59
atencion,0.52,0.16,3.26


**Qué es esa aceleración.** La forma habitual de aplicar una red a cada tránsito los recorre uno por
uno. Aquí los 32 tránsitos se pliegan dentro del lote y se procesan de una vez. Es la misma función
con los mismos pesos (hay una prueba automática que lo verifica); solo cambia la velocidad.

## 5. Cómo se juntan: la cabeza de fusión

Cada rama entrega un vector (32, 32 y 16 números). La cabeza de fusión los concatena con una bandera
por rama que dice si esa rama tenía dato, y pasa todo por `Densa(64) → Dropout(0,3) → Densa(3)`.
Las ramas quedan **congeladas**: la cabeza no las modifica. Hay un codificador por partición, de modo
que ningún ejemplo de prueba fue visto por el codificador que lo describe. El cuaderno 4 muestra el resultado.